In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

Feature Selection

In [2]:
# loading the dataset
df = pd.read_csv('hamon_googlefit_medical_realistic.csv')

# defining the feature categories
identifier_cols = ['user_id', 'day_index']
target_col = 'cardiometabolic_risk_state'

# mapping lifestyle and biomarker features
lifestyle_features = [
    'steps', 'calories_burned', 'calories_consumed', 'move_minutes',
    'distance_km', 'sleep_hours', 'sleep_efficiency', 'water_intake_l'
]
biomarker_features = [
    'avg_heart_rate', 'resting_hr', 'bp_systolic', 'bp_diastolic',
    'fasting_glucose', 'spo2', 'hrv', 'bmi'
]

# keeping only the features we need, in df
df = df[identifier_cols + lifestyle_features + biomarker_features + [target_col]].copy()

display(df.head())

,user_id,day_index,steps,calories_burned,calories_consumed,move_minutes,distance_km,sleep_hours,sleep_efficiency,water_intake_l,avg_heart_rate,resting_hr,bp_systolic,bp_diastolic,fasting_glucose,spo2,hrv,bmi,cardiometabolic_risk_state
0,1,1,10362,2038,2246,71,7.77,7.58540,0.93389,2.6,76,55,127.0,83.0,NaN,95,56,30.753735,2
1,1,2,11127,2337,2497,74,8.35,NaN,NaN,2.3,76,54,NaN,NaN,NaN,96,58,30.753735,2
2,1,3,11771,2098,2648,61,8.83,7.01264,0.91370,1.8,72,54,NaN,NaN,111.0,95,59,30.753735,2
3,1,4,12574,2135,2923,83,9.43,NaN,NaN,2.1,78,54,NaN,NaN,109.0,95,58,30.753735,2
4,1,5,10253,2058,2372,73,7.69,NaN,NaN,2.8,74,53,127.0,82.0,NaN,97,60,30.753735,2


Data Preprocessing

In [3]:
# sorting the data temporally(based on day index) to ensure correct time series sequence
df = df.sort_values(by=['user_id', 'day_index'])

# keeping only users with more than 2 days data(visits)
user_counts = df['user_id'].value_counts()
valid_users = user_counts[user_counts >= 2].index
df = df[df['user_id'].isin(valid_users)].copy()

# handling missing data
def impute_patient_data(group):
    # using interpolation for missing values in between
    group = group.interpolate(method='linear', limit_direction='both')
    # using forward fill for values missing at the end
    group = group.ffill()
    # using backfill for values missing at the start
    group = group.bfill()
    return group

# applying patient specific imputation
df_imputed = df.groupby('user_id', group_keys=False).apply(impute_patient_data)

# applying global mean fill (if entire feature was missing for a specific user)
global_means = df_imputed[lifestyle_features + biomarker_features].mean()
df_imputed[lifestyle_features + biomarker_features] = df_imputed[lifestyle_features + biomarker_features].fillna(global_means)


# data standardization (mean = 0, std dev = 1)
scaler = StandardScaler()
features_to_scale = lifestyle_features + biomarker_features
df_imputed[features_to_scale] = scaler.fit_transform(df_imputed[features_to_scale])


# padding for time series compatibility
max_timesteps = df_imputed['day_index'].nunique() # 30 for this dataset since 30 days
num_features = len(features_to_scale)

unique_users = df_imputed['user_id'].unique()
num_users = len(unique_users)

# initializing the 3D array with NaNs- making empty box
# shape: [Patients, Timesteps, Features]
X_padded = np.full((num_users, max_timesteps, num_features), np.nan)
Y_target = np.zeros(num_users) # array to contain the target classification

user_to_idx = {user: idx for idx, user in enumerate(unique_users)}

for user_id, group in df_imputed.groupby('user_id'):
    idx = user_to_idx[user_id]

    # Get the time-series data for this user
    user_features = group[features_to_scale].values
    time_steps_available = user_features.shape[0]

    # Insert into the padded array (padding at the end remains NaN)
    X_padded[idx, :time_steps_available, :] = user_features

    # Extract the target classification (assume static for the sequence)
    Y_target[idx] = group[target_col].iloc[0]

print(f"Preprocessing Complete")
print(f"Final 3D Time Series Shape (X): {X_padded.shape} -> [Patients, Timesteps, Features]")
print(f"Final Target Array Shape (Y): {Y_target.shape}")

/tmp/ipykernel_35256/3821135267.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_imputed = df.groupby('user_id', group_keys=False).apply(impute_patient_data)


Preprocessing Complete
Final 3D Time Series Shape (X): (3000, 30, 16) -> [Patients, Timesteps, Features]
Final Target Array Shape (Y): (3000,)


Visualizing the preprocessed data

In [4]:
print(f"Total Patients (Samples): {X_padded.shape[0]}")
print(f"Max Timesteps (Days):     {X_padded.shape[1]}")
print(f"Features per Timestep:    {X_padded.shape[2]}")

# inspecting the first user in the array (Index 0)
user_idx = 0
actual_user_id = unique_users[user_idx]

print(f"Target Classification Label (Y): {Y_target[user_idx]}")

# converting this user's 2D slice back into a dataFrame
user_0_df = pd.DataFrame(X_padded[user_idx], columns=features_to_scale)

print("\nFirst 5 Days (Standardized & Imputed)")
# values must look like decimals (e.g., 0.5, -1.2) because they are standardized
print(user_0_df.head(5).to_string())

print("\nLast 5 Days (Checking for NaN Padding)")
# if any user had fewer than 30 days of data, we must see 'NaN' values here
print(user_0_df.tail(5).to_string())


Total Patients (Samples): 3000
Max Timesteps (Days):     30
Features per Timestep:    16
Target Classification Label (Y): 2.0

First 5 Days (Standardized & Imputed)
      steps  calories_burned  calories_consumed  move_minutes  distance_km  sleep_hours  sleep_efficiency  water_intake_l  avg_heart_rate  resting_hr  bp_systolic  bp_diastolic  fasting_glucose      spo2       hrv       bmi
0  0.939814         0.869063          -0.712623      1.066107     0.939296     0.483875          0.877350        0.496988       -0.067594   -0.767503    -0.430188     -0.317057         0.564982 -0.396937  0.131125  1.065257
1  1.131496         2.366133           0.117289      1.178336     1.133066     0.238625          0.768186       -0.002644       -0.067594   -0.848700    -0.430188     -0.340948         0.564982  0.458224  0.231541  1.065257
2  1.292859         1.169479           0.616559      0.692012     1.293427    -0.006625          0.659022       -0.835365       -0.410237   -0.848700    -0.430188 

Time Series Forest Regressor

Using the first 10 days to predict the next 5 days data

In [5]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

# using the first 10 days to predict the next 5 days
OBSERVED_DAYS = 10
PREDICT_DAYS = 5

# The indices of the biomarkers we want to forecast (from Phase 1)
# biomarkers are the last 8 features before the target
num_lifestyle_features = 8
num_biomarker_features = 8
biomarker_start_idx = num_lifestyle_features
biomarker_end_idx = num_lifestyle_features + num_biomarker_features

# slicing the 3D Tensor for Forecasting
# filtering out users who don't have enough days of data
valid_user_indices = []
for i in range(X_padded.shape[0]):
    # check if the user has at least 15 days of non-NaN data
    if not np.isnan(X_padded[i, OBSERVED_DAYS + PREDICT_DAYS - 1, 0]):
        valid_user_indices.append(i)

X_valid = X_padded[valid_user_indices]
print(f"Valid users with at least {OBSERVED_DAYS + PREDICT_DAYS} days of data: {len(X_valid)}")

# X_input: All features (lifestyle + biomarkers) for the first 10 days
X_input = X_valid[:, :OBSERVED_DAYS, :]

# Y_target: Only the biomarkers for the next 5 days (Days 11 to 15)
Y_future = X_valid[:, OBSERVED_DAYS : OBSERVED_DAYS + PREDICT_DAYS, biomarker_start_idx:biomarker_end_idx]

# 2D input (samples, flattened features)
X_input_flat = X_input.reshape(X_input.shape[0], -1)
Y_future_flat = Y_future.reshape(Y_future.shape[0], -1)

X_train, X_test, y_train, y_test = train_test_split(X_input_flat, Y_future_flat, test_size=0.2, random_state=42)

# model training
# using 200 estimators
tsfr_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
tsfr_model.fit(X_train, y_train)

# model evaluation
y_pred = tsfr_model.predict(X_test)

# calculating standard metrics
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# calculating Normalized RMSE (NRMSE) as mentioned in the paper
# normalizing by the standard deviation of the true values
sigma_y = np.std(y_test) # calculates the std dev of the true data
nrmse = rmse / sigma_y if sigma_y != 0 else 0 # divides error of model with std dev

print(f"Forecasting Performance Metrics :")
print(f"MSE:   {mse:.4f}")
print(f"RMSE:  {rmse:.4f}")
print(f"MAE:   {mae:.4f}")
print(f"NRMSE: {nrmse:.4f} (Goal: < 1.0 for stable forecasting)")
print(f"R²:    {r2:.4f} (Goal: > 0.2 for reliable forecasting)")

# Example Output Generator:
# selecting random patient
sample_idx = 0
sample_input = X_test[sample_idx].reshape(1, -1)
sample_true_future = y_test[sample_idx].reshape(PREDICT_DAYS, num_biomarker_features)
sample_pred_future = y_pred[sample_idx].reshape(PREDICT_DAYS, num_biomarker_features)

# looking specifically at the first biomarker (eg- avg_heart_rate)
biomarker_idx_to_display = 0
biomarker_name = biomarker_features[biomarker_idx_to_display]

print(f"\nExample Forecasting Output for [{biomarker_name}]")
print(f"Patient {sample_idx} Trajectory (Days 11 to 15)")

output_comparison = pd.DataFrame({
    'Day': [f"Day {OBSERVED_DAYS + i + 1}" for i in range(PREDICT_DAYS)],
    'True_Value (Scaled)': sample_true_future[:, biomarker_idx_to_display],
    'Predicted_Value (Scaled)': sample_pred_future[:, biomarker_idx_to_display]
})

print(output_comparison.to_string(index=False))

Valid users with at least 15 days of data: 3000
Forecasting Performance Metrics :
MSE:   0.3468
RMSE:  0.5889
MAE:   0.4155
NRMSE: 0.5756 (Goal: < 1.0 for stable forecasting)
R²:    0.6659 (Goal: > 0.2 for reliable forecasting)

Example Forecasting Output for [avg_heart_rate]
Patient 0 Trajectory (Days 11 to 15)
   Day  True_Value (Scaled)  Predicted_Value (Scaled)
Day 11            -0.238915                  0.302460
Day 12            -0.495897                  0.129426
Day 13             0.018067                  0.353000
Day 14             0.874674                  0.379555
Day 15            -0.324576                  0.382981


Using the first 2 days to predict the next 5 days data (as in the paper)

In [6]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

# using the first 2 days to predict the next 5 days of biomarkers (7 days total)
OBSERVED_DAYS = 2
PREDICT_DAYS = 5

num_lifestyle_features = 8
num_biomarker_features = 8
biomarker_start_idx = num_lifestyle_features
biomarker_end_idx = num_lifestyle_features + num_biomarker_features

# slicing the 3D Tensor for Forecasting
valid_user_indices = []
for i in range(X_padded.shape[0]):
    # Check if the user has at least 7 days of non-NaN data
    if not np.isnan(X_padded[i, OBSERVED_DAYS + PREDICT_DAYS - 1, 0]):
        valid_user_indices.append(i)

X_valid = X_padded[valid_user_indices]

# X_input: Lifestyle + biomarkers for Days 1 and 2
X_input = X_valid[:, :OBSERVED_DAYS, :]

# Y_future: Only the biomarkers for Days 3, 4, and 5
Y_future = X_valid[:, OBSERVED_DAYS : OBSERVED_DAYS + PREDICT_DAYS, biomarker_start_idx:biomarker_end_idx]

# flattening for Tree based regressor
X_input_flat = X_input.reshape(X_input.shape[0], -1)
Y_future_flat = Y_future.reshape(Y_future.shape[0], -1)

X_train, X_test, y_train, y_test = train_test_split(X_input_flat, Y_future_flat, test_size=0.2, random_state=42)

# model training
tsfr_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
tsfr_model.fit(X_train, y_train)

# model evaluation
y_pred = tsfr_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# NRMSE Calculation
sigma_y = np.std(y_test)
nrmse = rmse / sigma_y if sigma_y != 0 else 0

print(f"Forecasting Performance Metrics")
print(f"RMSE:  {rmse:.4f}")
print(f"MAE:   {mae:.4f}")
print(f"NRMSE: {nrmse:.4f} (Goal: < 1.0)")
print(f"R²:    {r2:.4f} (Goal: > 0.2)")

# 5. Example Output Generator
sample_idx = 0
sample_true_future = y_test[sample_idx].reshape(PREDICT_DAYS, num_biomarker_features)
sample_pred_future = y_pred[sample_idx].reshape(PREDICT_DAYS, num_biomarker_features)

biomarker_idx_to_display = 0 # eg, avg_heart_rate
biomarker_name = biomarker_features[biomarker_idx_to_display]

print(f"\nExample Forecasting Output for [{biomarker_name}]")
print(f"Patient {sample_idx} Trajectory (Days 3 to 7)")

output_comparison = pd.DataFrame({
    'Day': [f"Day {OBSERVED_DAYS + i + 1}" for i in range(PREDICT_DAYS)],
    'True_Value': sample_true_future[:, biomarker_idx_to_display],
    'Predicted_Value': sample_pred_future[:, biomarker_idx_to_display]
})

print(output_comparison.to_string(index=False))

Forecasting Performance Metrics
RMSE:  0.5883
MAE:   0.4074
NRMSE: 0.6035 (Goal: < 1.0)
R²:    0.6418 (Goal: > 0.2)

Example Forecasting Output for [avg_heart_rate]
Patient 0 Trajectory (Days 3 to 7)
  Day  True_Value  Predicted_Value
Day 3    0.275049         0.288326
Day 4    0.103728         0.226222
Day 5   -0.324576         0.200524
Day 6    0.360710         0.217656
Day 7    0.360710         0.267339


Using the first 2 days to predict the next 3 days of data <br>
Observation : this performs better than the previous model in which we predicted 5 days data from just 2 days of data

In [7]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

# using the first 2 days to predict the next 3 days of biomarkers (5 days total)
OBSERVED_DAYS = 2
PREDICT_DAYS = 3

num_lifestyle_features = 8
num_biomarker_features = 8
biomarker_start_idx = num_lifestyle_features
biomarker_end_idx = num_lifestyle_features + num_biomarker_features

# slicing the 3D Tensor for Forecasting
valid_user_indices = []
for i in range(X_padded.shape[0]):
    # Check if the user has at least 5 days of non-NaN data
    if not np.isnan(X_padded[i, OBSERVED_DAYS + PREDICT_DAYS - 1, 0]):
        valid_user_indices.append(i)

X_valid = X_padded[valid_user_indices]

# X_input: Lifestyle + biomarkers for Days 1 and 2
X_input = X_valid[:, :OBSERVED_DAYS, :]

# Y_future: Only the biomarkers for Days 3, 4, 5
Y_future = X_valid[:, OBSERVED_DAYS : OBSERVED_DAYS + PREDICT_DAYS, biomarker_start_idx:biomarker_end_idx]

# flattening for Tree based regressor
X_input_flat = X_input.reshape(X_input.shape[0], -1)
Y_future_flat = Y_future.reshape(Y_future.shape[0], -1)

X_train, X_test, y_train, y_test = train_test_split(X_input_flat, Y_future_flat, test_size=0.2, random_state=42)

# model training
tsfr_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
tsfr_model.fit(X_train, y_train)

# model evaluation
y_pred = tsfr_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# NRMSE Calculation
sigma_y = np.std(y_test)
nrmse = rmse / sigma_y if sigma_y != 0 else 0

print(f"Forecasting Performance Metrics")
print(f"RMSE:  {rmse:.4f}")
print(f"MAE:   {mae:.4f}")
print(f"NRMSE: {nrmse:.4f} (Goal: < 1.0)")
print(f"R²:    {r2:.4f} (Goal: > 0.2)")

# 5. Example Output Generator
sample_idx = 0
sample_true_future = y_test[sample_idx].reshape(PREDICT_DAYS, num_biomarker_features)
sample_pred_future = y_pred[sample_idx].reshape(PREDICT_DAYS, num_biomarker_features)

biomarker_idx_to_display = 0 # eg, avg_heart_rate
biomarker_name = biomarker_features[biomarker_idx_to_display]

print(f"\nExample Forecasting Output for [{biomarker_name}]")
print(f"Patient {sample_idx} Trajectory (Days 3 to 7)")

output_comparison = pd.DataFrame({
    'Day': [f"Day {OBSERVED_DAYS + i + 1}" for i in range(PREDICT_DAYS)],
    'True_Value': sample_true_future[:, biomarker_idx_to_display],
    'Predicted_Value': sample_pred_future[:, biomarker_idx_to_display]
})

print(output_comparison.to_string(index=False))

Forecasting Performance Metrics
RMSE:  0.5574
MAE:   0.3823
NRMSE: 0.5785 (Goal: < 1.0)
R²:    0.6748 (Goal: > 0.2)

Example Forecasting Output for [avg_heart_rate]
Patient 0 Trajectory (Days 3 to 7)
  Day  True_Value  Predicted_Value
Day 3    0.275049         0.302460
Day 4    0.103728         0.252777
Day 5   -0.324576         0.267339


Comparing the resuls with the Naive Baseline Model (LOCF)

In [8]:
print("NAIVE BASELINE (Last Observation Carried Forward)")

# Extract the last observed day (Day 2) biomarkers from the flattened X_test
total_features = num_lifestyle_features + num_biomarker_features

# Day 2's features are at the end of the flattened array
start_idx = (OBSERVED_DAYS - 1) * total_features + biomarker_start_idx
end_idx = start_idx + num_biomarker_features

last_observed_biomarkers = X_test[:, start_idx:end_idx]

# Repeating Day 2's biomarkers to guess Day 3, 4, and 5
# np.tile copies the Day 2 array 3 times in a row to match the flattened y_test shape
naive_y_pred = np.tile(last_observed_biomarkers, (1, PREDICT_DAYS))

# Evaluating the Naive Baseline
naive_mse = mean_squared_error(y_test, naive_y_pred)
naive_rmse = np.sqrt(naive_mse)
naive_mae = mean_absolute_error(y_test, naive_y_pred)
naive_r2 = r2_score(y_test, naive_y_pred)

naive_sigma_y = np.std(y_test)
naive_nrmse = naive_rmse / naive_sigma_y if naive_sigma_y != 0 else 0

print(f"Naive LOCF Metrics")
print(f"RMSE:  {naive_rmse:.4f}")
print(f"MAE:   {naive_mae:.4f}")
print(f"NRMSE: {naive_nrmse:.4f}")
print(f"R²:    {naive_r2:.4f}")

# 4. Final Showdown
print("MODEL COMPARISON: Random Forest vs Naive Baseline")
print(f"Random Forest MAE: {mae:.4f}  |  Naive MAE: {naive_mae:.4f}")
print(f"Random Forest R²:  {r2:.4f}  |  Naive R²:  {naive_r2:.4f}")

if mae < naive_mae:
    print("\nSUCCESS: Your Random Forest is capturing temporal trends and beating the baseline!")
else:
    print("\nNOTE: The baseline won. The Random Forest might be struggling with the sparse 2-day context.")

NAIVE BASELINE (Last Observation Carried Forward)
Naive LOCF Metrics
RMSE:  0.6302
MAE:   0.3489
NRMSE: 0.6541
R²:    0.5934
MODEL COMPARISON: Random Forest vs Naive Baseline
Random Forest MAE: 0.3823  |  Naive MAE: 0.3489
Random Forest R²:  0.6748  |  Naive R²:  0.5934

NOTE: The baseline won. The Random Forest might be struggling with the sparse 2-day context.


LSTM Model

In [9]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Reshape, Dropout, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

# 10 Days to 5 Days
OBSERVED_DAYS = 10
PREDICT_DAYS = 5

num_features_in = X_padded.shape[2]
num_features_out = 8

# Slicing the 3d tensor
valid_user_indices = []
for i in range(X_padded.shape[0]):
    # Ensure patient has at least 15 days of data (10 observed + 5 predicted)
    if not np.isnan(X_padded[i, OBSERVED_DAYS + PREDICT_DAYS - 1, 0]):
        valid_user_indices.append(i)

X_valid = X_padded[valid_user_indices]

# Slice Y_target to match the valid users
Y_target_valid = Y_target[valid_user_indices]

# X_input: Keep 3D shape -> [Patients, 10 Days, Features]
X_input = X_valid[:, :OBSERVED_DAYS, :]

# Y_future: 3D shape -> [Patients, 5 Days, 8 Biomarkers]
Y_future = X_valid[:, OBSERVED_DAYS : OBSERVED_DAYS + PREDICT_DAYS, -num_features_out:]

# Split X_input, Y_future, AND Y_target_valid all at once
X_train, X_test, y_train, y_test, y_train_labels, y_test_labels = train_test_split(
    X_input, Y_future, Y_target_valid, test_size=0.2, random_state=42
)

# LSTM architecture: Deeper and Bidirectional
model = Sequential([
    Input(shape=(OBSERVED_DAYS, num_features_in)),

    # First Bidirectional LSTM layer
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.3),

    # Second LSTM layer to capture higher-level patterns
    LSTM(64, return_sequences=False),

    Dense(64, activation='relu'),
    Dense(PREDICT_DAYS * num_features_out),
    Reshape((PREDICT_DAYS, num_features_out))
])

# Using a lower learning rate to avoid the mean trap
optimizer = Adam(learning_rate=0.0005)
model.compile(optimizer=optimizer, loss='mae')
model.summary()

# Model training: More epochs with EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

# Model evaluation
print("\nEvaluating Model Predictions on Test Set...")
y_pred = model.predict(X_test)

# Flattening for metric calculation
y_test_flat = y_test.reshape(y_test.shape[0], -1)
y_pred_flat = y_pred.reshape(y_pred.shape[0], -1)

mse = mean_squared_error(y_test_flat, y_pred_flat)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_flat, y_pred_flat)
r2 = r2_score(y_test_flat, y_pred_flat)

# NRMSE Calculation
sigma_y = np.std(y_test_flat)
nrmse = rmse / sigma_y if sigma_y != 0 else 0

print(f"\n--- LSTM Forecasting Metrics (10-Day Context) ---")
print(f"MSE:   {mse:.4f}")
print(f"RMSE:  {rmse:.4f}")
print(f"MAE:   {mae:.4f}")
print(f"NRMSE: {nrmse:.4f} (Goal: < 1.0)")
print(f"R²:    {r2:.4f} (Goal: > 0.2)")

# Example Output Generator
sample_idx = 0
sample_true_future = y_test[sample_idx]
sample_pred_future = y_pred[sample_idx]

# Assuming biomarker_features list was defined in Phase 1
biomarker_idx_to_display = 0 # e.g., avg_heart_rate
biomarker_name = biomarker_features[biomarker_idx_to_display]

print(f"\nExample LSTM Output for [{biomarker_name}]")
print(f"Patient {sample_idx} Trajectory (Days 11 to 15)")

output_comparison = pd.DataFrame({
    'Day': [f"Day {OBSERVED_DAYS + i + 1}" for i in range(PREDICT_DAYS)],
    'True_Value': sample_true_future[:, biomarker_idx_to_display],
    'LSTM_Predicted': sample_pred_future[:, biomarker_idx_to_display]
})

print(output_comparison.to_string(index=False))

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 10, 256)        │       148,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 10, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        82,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 40)             │         2,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 5, 8)           │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 237,416 (927.41 KB)

 Trainable params: 237,416 (927.41 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - loss: 0.6104 - val_loss: 0.4696
Epoch 2/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.4387 - val_loss: 0.4077
Epoch 3/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.4001 - val_loss: 0.3871
Epoch 4/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.3875 - val_loss: 0.3782
Epoch 5/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.3793 - val_loss: 0.3765
Epoch 6/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.3736 - val_loss: 0.3713
Epoch 7/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.3709 - val_loss: 0.3716
Epoch 8/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - loss: 0.3659 - val_loss: 0.3647
Epoch 9/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 0.3631 - val_loss: 0.3606
Epoch 10/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.3608 - val_loss: 0.3634
Epoch 11/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.3584 - val_loss: 0.3597
Epoch 12/100
120/120 ━━━━━━━━━━━━━━━━━

In [10]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Bidirectional, LSTM, Dense, Dropout, Reshape
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 2 Days to 3 Days
OBSERVED_DAYS = 2
PREDICT_DAYS = 5

num_features_in = X_padded.shape[2]
num_features_out = 8

# slicing the 3d tensor
valid_user_indices = []
for i in range(X_padded.shape[0]):
    # Ensure patient has at least 5 days of data (2 observed + 3 predicted)
    if not np.isnan(X_padded[i, OBSERVED_DAYS + PREDICT_DAYS - 1, 0]):
        valid_user_indices.append(i)

X_valid = X_padded[valid_user_indices]

# THE MISSING PIECE: We must also slice Y_target to match the valid users!
Y_target_valid = Y_target[valid_user_indices]

# X_input: Keep 3D shape -> [Patients, 2 Days, Features]
X_input = X_valid[:, :OBSERVED_DAYS, :]

# Y_future: 3D shape -> [Patients, 3 Days, 8 Biomarkers]
Y_future = X_valid[:, OBSERVED_DAYS : OBSERVED_DAYS + PREDICT_DAYS, -num_features_out:]

# ---> UPDATED SPLIT: Now we split X_input, Y_future, AND Y_target_valid all at once!
X_train, X_test, y_train, y_test, y_train_labels, y_test_labels = train_test_split(
    X_input, Y_future, Y_target_valid, test_size=0.2, random_state=42
)

# LSTM architecture
model = Sequential([
    Input(shape=(OBSERVED_DAYS, num_features_in)),

    # Since the sequence is short (2 days), a Bidirectional layer is highly effective
    # at capturing the immediate transition/velocity between Day 1 and Day 2.
    Bidirectional(LSTM(64, return_sequences=False)),

    Dense(64, activation='relu'),
    Dropout(0.2),

    Dense(PREDICT_DAYS * num_features_out),
    Reshape((PREDICT_DAYS, num_features_out))
])

optimizer = Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss='mae')
model.summary()

# model training
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

# model evaluation
y_pred = model.predict(X_test)

y_test_flat = y_test.reshape(-1, 1)
y_pred_flat = y_pred.reshape(-1, 1)

print(f"\nLSTM Forecasting Metrics (2-Day Context)")
print(f"R²:    {r2_score(y_test_flat, y_pred_flat):.4f}")
print(f"NRMSE: {np.sqrt(mean_squared_error(y_test_flat, y_pred_flat)) / np.std(y_test_flat):.4f}")

# Example Output
sample_idx = 0
print(f"\nTrajectory for [avg_heart_rate] (Days 3-5):")
comparison = pd.DataFrame({
    'True_Value': y_test[sample_idx, :, 0],
    'LSTM_Predicted': y_pred[sample_idx, :, 0]
})
print(comparison)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_1 (Bidirectional) │ (None, 128)            │        41,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 40)             │         2,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_1 (Reshape)             │ (None, 5, 8)           │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 52,328 (204.41 KB)

 Trainable params: 52,328 (204.41 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - loss: 0.5856 - val_loss: 0.4328
Epoch 2/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - loss: 0.4366 - val_loss: 0.3796
Epoch 3/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 6s 36ms/step - loss: 0.4129 - val_loss: 0.3664
Epoch 4/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 5s 37ms/step - loss: 0.4007 - val_loss: 0.3614
Epoch 5/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - loss: 0.3937 - val_loss: 0.3641
Epoch 6/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3903 - val_loss: 0.3555
Epoch 7/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3856 - val_loss: 0.3518
Epoch 8/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.3851 - val_loss: 0.3593
Epoch 9/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3793 - val_loss: 0.3559
Epoch 10/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3787 - val_loss: 0.3533
Epoch 11/100
120/120 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.3761 - val_loss: 0.3535
Epoch 12/100
120/120 ━━━━━━━━━━━━━━━

Comparing the resuls with the Naive Baseline model (LOCF)

In [11]:
print("NAIVE BASELINE (Last Observation Carried Forward)")

# Generating Naive Predictions
# Take the exact biomarker values from the last observed day (index -1)
# X_test shape is [Samples, OBSERVED_DAYS, Features]
# Biomarkers are the last 'num_features_out' columns
last_observed_day = X_test[:, -1, -num_features_out:]

# Repeat that single day's values across all future PREDICT_DAYS
naive_y_pred = np.repeat(last_observed_day[:, np.newaxis, :], PREDICT_DAYS, axis=1)

# Evaluating Naive Predictions against the Ground Truth (y_test)
naive_y_test_flat = y_test.reshape(-1, 1)
naive_y_pred_flat = naive_y_pred.reshape(-1, 1)

naive_mse = mean_squared_error(naive_y_test_flat, naive_y_pred_flat)
naive_rmse = np.sqrt(naive_mse)
naive_mae = mean_absolute_error(naive_y_test_flat, naive_y_pred_flat)
naive_r2 = r2_score(naive_y_test_flat, naive_y_pred_flat)

naive_sigma_y = np.std(naive_y_test_flat)
naive_nrmse = naive_rmse / naive_sigma_y if naive_sigma_y != 0 else 0

print(f"Naive LOCF Metrics")
print(f"MSE:   {naive_mse:.4f}")
print(f"RMSE:  {naive_rmse:.4f}")
print(f"MAE:   {naive_mae:.4f}")
print(f"NRMSE: {naive_nrmse:.4f}")
print(f"R²:    {naive_r2:.4f}")

# 3. Direct Comparison
print("\nModel Comparison (LSTM vs Naive)")
print(f"LSTM MAE:  {mae:.4f}  |  Naive MAE:  {naive_mae:.4f}")
print(f"LSTM R²:   {r2:.4f}  |  Naive R²:   {naive_r2:.4f}")

if mae < naive_mae:
    print("\nSUCCESS: Your LSTM is capturing temporal dynamics and outperforming the Naive Baseline!")
else:
    print("\nWARNING: Your LSTM is performing worse than doing nothing. The model is likely underfitting or suffering from mode collapse.")

NAIVE BASELINE (Last Observation Carried Forward)
Naive LOCF Metrics
MSE:   0.4456
RMSE:  0.6675
MAE:   0.3877
NRMSE: 0.6848
R²:    0.5311

Model Comparison (LSTM vs Naive)
LSTM MAE:  0.3641  |  Naive MAE:  0.3877
LSTM R²:   0.7003  |  Naive R²:   0.5311

SUCCESS: Your LSTM is capturing temporal dynamics and outperforming the Naive Baseline!


Classification (1D-CNN)

In [12]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, GlobalMaxPooling1D, Dense, Dropout, Input
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split

print("         PHASE 3: CLASSIFICATION             ")

# Defining Common Variables
total_days = OBSERVED_DAYS + PREDICT_DAYS  # 7 days
total_features = X_test.shape[2]           # 16 features


# SCENARIO 1: GROUND TRUTH
print("\nSCENARIO 1: GROUND TRUTH (100% Real Data)")

# extract true 7-day sequences for the exact same patients
X_full_5_days = X_valid[:, :OBSERVED_DAYS + PREDICT_DAYS, :]

_, X_test_ground_truth, _, _ = train_test_split(
    X_full_5_days, Y_target_valid, test_size=0.2, random_state=42
)

# building Ground Truth CNN
model_cnn_gt = Sequential([
    Input(shape=(total_days, total_features)),
    Conv1D(filters=64, kernel_size=2, activation='relu'),
    Dropout(0.3),
    GlobalMaxPooling1D(),
    Dense(32, activation='relu'),
    Dense(5, activation='softmax')
])

model_cnn_gt.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# training CNN on Pure Ground Truth Data
model_cnn_gt.fit(X_test_ground_truth, y_test_labels, epochs=50, batch_size=16, validation_split=0.2, verbose=0)

y_pred_probs_gt = model_cnn_gt.predict(X_test_ground_truth)
y_pred_classes_gt = np.argmax(y_pred_probs_gt, axis=1)
gt_acc = accuracy_score(y_test_labels, y_pred_classes_gt)


# SCENARIO 2: HYBRID (Real + Forecasted Data)

print("\nSCENARIO 2: HYBRID (2 Real + 5 Forecasted Days)")

# stitching procedure
num_bio_features = y_pred.shape[2]
num_non_bio_features = total_features - num_bio_features

non_bio_means = X_test[:, :, :num_non_bio_features].mean(axis=1, keepdims=True)
non_bio_forecasted = np.repeat(non_bio_means, PREDICT_DAYS, axis=1)

forecasted_sequence = np.concatenate([non_bio_forecasted, y_pred], axis=2)
full_stitched_sequence = np.concatenate([X_test, forecasted_sequence], axis=1)

# building Hybrid CNN
model_cnn_hybrid = Sequential([
    Input(shape=(total_days, total_features)),
    Conv1D(filters=64, kernel_size=2, activation='relu'),
    Dropout(0.3),
    GlobalMaxPooling1D(),
    Dense(32, activation='relu'),
    Dense(5, activation='softmax')
])

model_cnn_hybrid.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# sraining CNN on stitched Data
model_cnn_hybrid.fit(full_stitched_sequence, y_test_labels, epochs=50, batch_size=16, validation_split=0.2, verbose=0)

y_pred_probs_hybrid = model_cnn_hybrid.predict(full_stitched_sequence)
y_pred_classes_hybrid = np.argmax(y_pred_probs_hybrid, axis=1)
hybrid_acc = accuracy_score(y_test_labels, y_pred_classes_hybrid)


# FINAL COMPARISON
print("\n")
print("          FINAL CLASSIFICATION RESULTS       ")
print(f"Ground Truth Scenario Accuracy (Gold Standard): {gt_acc:.4f}")
print(f"Hybrid Scenario Accuracy (LSTM Forecasted):     {hybrid_acc:.4f}")

if hybrid_acc >= gt_acc * 0.90:
    print("\nSUCCESS: Your forecasting model is highly reliable! The Hybrid pipeline retains over 90% of the Ground Truth accuracy.")
else:
    print("\nNOTE: The Hybrid model is losing some accuracy compared to Ground Truth. Improving the LSTM forecast will close this gap.")



         PHASE 3: CLASSIFICATION             

SCENARIO 1: GROUND TRUTH (100% Real Data)
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step

SCENARIO 2: HYBRID (2 Real + 5 Forecasted Days)
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step


          FINAL CLASSIFICATION RESULTS       
Ground Truth Scenario Accuracy (Gold Standard): 0.9283
Hybrid Scenario Accuracy (LSTM Forecasted):     0.9417

SUCCESS: Your forecasting model is highly reliable! The Hybrid pipeline retains over 90% of the Ground Truth accuracy.


In [13]:


# METRICS TABLE (Precision, Recall, F1)
print("Detailed Classification Metrics (Hybrid Model)")
# getting the metrics as a structured dictionary
report_dict = classification_report(y_test_labels, y_pred_classes_hybrid, output_dict=True, zero_division=0)
# converting the dictionary into a Pandas DataFrame and take transpose
report_df = pd.DataFrame(report_dict).transpose()
report_df = report_df.drop(['accuracy', 'macro avg', 'weighted avg'], errors='ignore')
# selecting only the columns we care about and round them to 4 decimal places
metrics_df = report_df[['precision', 'recall', 'f1-score']].round(4)
# cleaning up the row names so it looks nice
metrics_df.index.name = 'Risk State'
metrics_df.reset_index(inplace=True)
# printing the table
print(metrics_df.to_string(index=False))


Detailed Classification Metrics (Hybrid Model)
Risk State  precision  recall  f1-score
       1.0     0.9730  0.8060    0.8816
       2.0     0.9368  0.9834    0.9595
       3.0     0.8750  0.9333    0.9032
       4.0     1.0000  1.0000    1.0000
